In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

mutation_matrix_path = Path(
    "../data/processed/gbm_primary_profiled_mutation_matrix.tsv"
)

mutation_matrix = pd.read_csv(
    mutation_matrix_path,
    sep="\t",
    index_col=0
)

print("Mutation matrix shape:",
      mutation_matrix.shape)

print("Unique values:",
      np.unique(mutation_matrix.to_numpy()))

Mutation matrix shape: (390, 13909)
Unique values: [0 1]


In [2]:
clinical_path = Path(
    "../data/processed/gbm_primary_clinical.tsv"
)

primary_clinical = pd.read_csv(
    clinical_path,
    sep="\t"
)

primary_sample_ids = set(
    primary_clinical["Sample ID"]
)

sequenced_path = Path(
    "../data/raw/gbm_tcga_pan_can_atlas_2018_cases_sequenced.txt"
)

lines = sequenced_path.read_text().splitlines()

case_ids_line = next(
    line for line in lines
    if line.startswith("case_list_ids:")
)

sequenced_sample_ids = set(
    case_ids_line
    .split(":", 1)[1]
    .strip()
    .split("\t")
)

expression_path = Path(
    "../data/raw/gbm_tcga_pan_can_atlas_2018_mrna_rsem.txt"
)

expression = pd.read_csv(
    expression_path,
    sep="\t",
    low_memory=False
)

expression_sample_ids = set(
    expression.columns[2:]
)

multiomics_primary_ids = sorted(
    primary_sample_ids
    & sequenced_sample_ids
    & expression_sample_ids
)

print(
    "Multi-omics primary patients:",
    len(multiomics_primary_ids)
)

Multi-omics primary patients: 152


In [3]:
mutation_multiomics = mutation_matrix.loc[
    multiomics_primary_ids
].copy()

print(
    "Mutation matrix for multi-omics cohort:",
    mutation_multiomics.shape
)

print(
    "Missing requested patients:",
    len(
        set(multiomics_primary_ids)
        - set(mutation_matrix.index)
    )
)

Mutation matrix for multi-omics cohort: (152, 13909)
Missing requested patients: 0


In [4]:
mutation_multiomics_t = (
    mutation_multiomics.T
)

print(
    "Transposed mutation matrix:",
    mutation_multiomics_t.shape
)

print(
    "Patients:",
    mutation_multiomics_t.shape[1]
)

print(
    "Genes:",
    mutation_multiomics_t.shape[0]
)

Transposed mutation matrix: (13909, 152)
Patients: 152
Genes: 13909


In [5]:
# Re-create expression_status from expression data for downstream integration
expression_multiomics = expression[
    ["Hugo_Symbol", "Entrez_Gene_Id"] + multiomics_primary_ids
]

expression_with_symbols = expression_multiomics[
    expression_multiomics["Hugo_Symbol"].notna()
    & (expression_multiomics["Hugo_Symbol"] != "")
]

expression_gene_matrix = (
    expression_with_symbols
    .groupby("Hugo_Symbol")[multiomics_primary_ids]
    .mean()
)

expression_log2 = np.log2(expression_gene_matrix + 1)
unique_values_per_gene = expression_log2.nunique(axis=1)
constant_genes = unique_values_per_gene[unique_values_per_gene == 1].index
expression_log2_filtered = expression_log2.drop(index=constant_genes)

gene_median = expression_log2_filtered.median(axis=1)
gene_mad = (
    expression_log2_filtered
    .sub(gene_median, axis=0)
    .abs()
    .median(axis=1)
)

nonzero_mad_genes = gene_mad[gene_mad > 0].index
expression_for_scoring = expression_log2_filtered.loc[nonzero_mad_genes]
gene_median_nonzero = gene_median.loc[nonzero_mad_genes]
gene_mad_nonzero = gene_mad.loc[nonzero_mad_genes]

expression_robust_z = (
    expression_for_scoring
    .sub(gene_median_nonzero, axis=0)
    .div(gene_mad_nonzero, axis=0)
    * 0.6745
)

expression_delta_log2 = (
    expression_for_scoring
    .sub(gene_median_nonzero, axis=0)
)

final_high_expression_mask = (
    (expression_robust_z >= 3.5)
    & (expression_delta_log2 >= 1.0)
)

final_low_expression_mask = (
    (expression_robust_z <= -3.5)
    & (expression_delta_log2 <= -1.0)
)

expression_status = pd.DataFrame(
    0,
    index=expression_robust_z.index,
    columns=expression_robust_z.columns,
    dtype="int8"
)

expression_status[final_high_expression_mask] = 1
expression_status[final_low_expression_mask] = -1

print("Expression status shape:", expression_status.shape)

Expression status shape: (17639, 152)


In [6]:
mutation_genes = set(
    mutation_multiomics_t.index
)

expression_genes = set(
    expression_status.index
)

shared_genes = (
    mutation_genes
    & expression_genes
)

mutation_only_genes = (
    mutation_genes
    - expression_genes
)

expression_only_genes = (
    expression_genes
    - mutation_genes
)

print(
    "Mutation genes:",
    len(mutation_genes)
)

print(
    "Expression-scored genes:",
    len(expression_genes)
)

print(
    "Shared genes:",
    len(shared_genes)
)

print(
    "Mutation-only genes:",
    len(mutation_only_genes)
)

print(
    "Expression-only genes:",
    len(expression_only_genes)
)

Mutation genes: 13909
Expression-scored genes: 17639
Shared genes: 11568
Mutation-only genes: 2341
Expression-only genes: 6071


In [7]:
all_integrated_genes = sorted(
    mutation_genes | expression_genes
)

print(
    "Total integrated genes:",
    len(all_integrated_genes)
)

Total integrated genes: 19980


In [8]:
mutation_evidence = (
    mutation_multiomics_t
    .reindex(
        index=all_integrated_genes,
        columns=multiomics_primary_ids,
        fill_value=0
    )
)

print(
    "Mutation evidence shape:",
    mutation_evidence.shape
)

print(
    "Mutation unique values:",
    np.unique(
        mutation_evidence.to_numpy()
    )
)

Mutation evidence shape: (19980, 152)
Mutation unique values: [0 1]


In [9]:
expression_evidence = (
    expression_status
    .reindex(
        index=all_integrated_genes,
        columns=multiomics_primary_ids,
        fill_value=0
    )
)

print(
    "Expression evidence shape:",
    expression_evidence.shape
)

print(
    "Expression unique values:",
    np.unique(
        expression_evidence.to_numpy()
    )
)

Expression evidence shape: (19980, 152)
Expression unique values: [-1  0  1]


In [10]:
expression_z_evidence = (
    expression_robust_z
    .reindex(
        index=all_integrated_genes,
        columns=multiomics_primary_ids
    )
)

expression_delta_evidence = (
    expression_delta_log2
    .reindex(
        index=all_integrated_genes,
        columns=multiomics_primary_ids
    )
)

print(
    "Robust Z evidence shape:",
    expression_z_evidence.shape
)

print(
    "Delta evidence shape:",
    expression_delta_evidence.shape
)

print(
    "Genes without robust expression score:",
    expression_z_evidence.isna().all(axis=1).sum()
)

Robust Z evidence shape: (19980, 152)
Delta evidence shape: (19980, 152)
Genes without robust expression score: 2341


In [11]:
print(
    "Same gene index:",
    mutation_evidence.index.equals(
        expression_evidence.index
    )
)

print(
    "Same patient columns:",
    mutation_evidence.columns.equals(
        expression_evidence.columns
    )
)

print(
    "Same Z-score layout:",
    mutation_evidence.index.equals(
        expression_z_evidence.index
    )
    and
    mutation_evidence.columns.equals(
        expression_z_evidence.columns
    )
)

Same gene index: True
Same patient columns: True
Same Z-score layout: True


In [12]:
example_patient = multiomics_primary_ids[0]

example_integrated = pd.DataFrame({
    "Mutation":
        mutation_evidence[
            example_patient
        ],

    "Expression_Status":
        expression_evidence[
            example_patient
        ],

    "Robust_Z":
        expression_z_evidence[
            example_patient
        ],

    "Delta_log2":
        expression_delta_evidence[
            example_patient
        ]
})

example_integrated_altered = (
    example_integrated[
        (example_integrated["Mutation"] == 1)
        |
        (example_integrated["Expression_Status"] != 0)
    ]
)

print(
    "Altered genes for example patient:",
    len(example_integrated_altered)
)

example_integrated_altered.head(20)

Altered genes for example patient: 123


,Mutation,Expression_Status,Robust_Z,Delta_log2
ABR,1,0,1.289862,0.761017
ADAMTS2,1,0,1.059058,1.772495
ADAMTS7,1,0,1.147008,1.321317
ARFIP2,0,-1,-3.675166,-1.202149
ARSG,1,0,0.041781,0.032029
ASAH1,1,0,0.388661,0.177317
ATP6V0A4,0,1,4.082734,6.256458
AVPR1A,0,1,4.311094,5.015222
BCL11A,1,0,0.384195,0.554397
BTBD3,1,0,2.405598,1.367487


In [13]:
mutation_positive = (
    mutation_evidence == 1
)

expression_positive = (
    expression_evidence != 0
)

both_evidence = (
    mutation_positive
    &
    expression_positive
)

mutation_only_evidence = (
    mutation_positive
    &
    ~expression_positive
)

expression_only_evidence = (
    ~mutation_positive
    &
    expression_positive
)

print(
    "Mutation-only gene-patient alterations:",
    mutation_only_evidence.sum().sum()
)

print(
    "Expression-only gene-patient alterations:",
    expression_only_evidence.sum().sum()
)

print(
    "Both mutation + expression alterations:",
    both_evidence.sum().sum()
)

Mutation-only gene-patient alterations: 15891
Expression-only gene-patient alterations: 22290
Both mutation + expression alterations: 201


In [14]:
integrated_altered_mask = (
    (mutation_evidence == 1)
    |
    (expression_evidence != 0)
)

print(
    "Total altered patient-gene pairs:",
    integrated_altered_mask.sum().sum()
)

Total altered patient-gene pairs: 38382


In [15]:
gene_positions, patient_positions = np.where(
    integrated_altered_mask.to_numpy()
)

integrated_altered_long = pd.DataFrame({
    "Hugo_Symbol":
        mutation_evidence.index.to_numpy()[
            gene_positions
        ],

    "Sample_ID":
        mutation_evidence.columns.to_numpy()[
            patient_positions
        ],

    "Mutation":
        mutation_evidence.to_numpy()[
            gene_positions,
            patient_positions
        ],

    "Expression_Status":
        expression_evidence.to_numpy()[
            gene_positions,
            patient_positions
        ],

    "Robust_Z":
        expression_z_evidence.to_numpy()[
            gene_positions,
            patient_positions
        ],

    "Delta_log2":
        expression_delta_evidence.to_numpy()[
            gene_positions,
            patient_positions
        ]
})

print(
    "Integrated altered table shape:",
    integrated_altered_long.shape
)

integrated_altered_long.head()

Integrated altered table shape: (38382, 6)


,Hugo_Symbol,Sample_ID,Mutation,Expression_Status,Robust_Z,Delta_log2
0,A1BG,TCGA-06-5411-01,1,0,0.796732,0.852444
1,A1CF,TCGA-06-2558-01,1,0,NaN,NaN
2,A2M,TCGA-06-0125-01,1,0,-0.845965,-0.734574
3,A2M,TCGA-06-5416-01,1,0,-1.254293,-1.089136
4,A2ML1,TCGA-06-5416-01,1,0,-1.399420,-3.763578


In [16]:
conditions = [
    (
        (integrated_altered_long["Mutation"] == 1)
        &
        (integrated_altered_long["Expression_Status"] == 0)
    ),

    (
        (integrated_altered_long["Mutation"] == 0)
        &
        (integrated_altered_long["Expression_Status"] == 1)
    ),

    (
        (integrated_altered_long["Mutation"] == 0)
        &
        (integrated_altered_long["Expression_Status"] == -1)
    ),

    (
        (integrated_altered_long["Mutation"] == 1)
        &
        (integrated_altered_long["Expression_Status"] == 1)
    ),

    (
        (integrated_altered_long["Mutation"] == 1)
        &
        (integrated_altered_long["Expression_Status"] == -1)
    )
]

choices = [
    "mutation_only",
    "expression_high_only",
    "expression_low_only",
    "mutation_plus_high",
    "mutation_plus_low"
]

integrated_altered_long["Evidence_Class"] = np.select(
    conditions,
    choices,
    default="unknown"
)

print(integrated_altered_long[
    "Evidence_Class"
].value_counts())

print(
    "Unknown classifications:",
    (
        integrated_altered_long[
            "Evidence_Class"
        ] == "unknown"
    ).sum()
)

print(
    "Total classified rows:",
    len(integrated_altered_long)
)

Evidence_Class
mutation_only           15891
expression_low_only     11882
expression_high_only    10408
mutation_plus_low         116
mutation_plus_high         85
Name: count, dtype: int64


Unknown classifications: 0
Total classified rows: 38382


In [17]:
example_patient_table = (
    integrated_altered_long[
        integrated_altered_long["Sample_ID"]
        == example_patient
    ]
)

print(
    "Example patient:",
    example_patient
)

print(
    "Altered genes:",
    len(example_patient_table)
)

print()
print(
    example_patient_table[
        "Evidence_Class"
    ].value_counts()
)

example_patient_table.head(20)

Example patient: TCGA-02-0047-01
Altered genes: 123

Evidence_Class
mutation_only           60
expression_high_only    49
expression_low_only     13
mutation_plus_low        1
Name: count, dtype: int64


,Hugo_Symbol,Sample_ID,Mutation,Expression_Status,Robust_Z,Delta_log2,Evidence_Class
254,ABR,TCGA-02-0047-01,1,0,1.289862,0.761017,mutation_only
579,ADAMTS2,TCGA-02-0047-01,1,0,1.059058,1.772495,mutation_only
601,ADAMTS7,TCGA-02-0047-01,1,0,1.147008,1.321317,mutation_only
1786,ARFIP2,TCGA-02-0047-01,0,-1,-3.675166,-1.202149,expression_low_only
2101,ARSG,TCGA-02-0047-01,1,0,0.041781,0.032029,mutation_only
2121,ASAH1,TCGA-02-0047-01,1,0,0.388661,0.177317,mutation_only
2472,ATP6V0A4,TCGA-02-0047-01,0,1,4.082734,6.256458,expression_high_only
2636,AVPR1A,TCGA-02-0047-01,0,1,4.311094,5.015222,expression_high_only
2904,BCL11A,TCGA-02-0047-01,1,0,0.384195,0.554397,mutation_only
3343,BTBD3,TCGA-02-0047-01,1,0,2.405598,1.367487,mutation_only


In [18]:
print(
    "Patients represented:",
    integrated_altered_long[
        "Sample_ID"
    ].nunique()
)

print(
    "Unique altered genes:",
    integrated_altered_long[
        "Hugo_Symbol"
    ].nunique()
)

Patients represented: 152
Unique altered genes: 15037


In [19]:
integrated_output_path = Path(
    "../data/processed/"
    "gbm_integrated_altered_gene_evidence.tsv"
)

integrated_altered_long.to_csv(
    integrated_output_path,
    sep="\t",
    index=False
)

print(
    "Saved:",
    integrated_output_path
)

print(
    "Exists:",
    integrated_output_path.exists()
)

integrated_check = pd.read_csv(
    integrated_output_path,
    sep="\t"
)

print(
    "Reloaded shape:",
    integrated_check.shape
)

integrated_check.head()

Saved: ..\data\processed\gbm_integrated_altered_gene_evidence.tsv
Exists: True
Reloaded shape: (38382, 7)


,Hugo_Symbol,Sample_ID,Mutation,Expression_Status,Robust_Z,Delta_log2,Evidence_Class
0,A1BG,TCGA-06-5411-01,1,0,0.796732,0.852444,mutation_only
1,A1CF,TCGA-06-2558-01,1,0,NaN,NaN,mutation_only
2,A2M,TCGA-06-0125-01,1,0,-0.845965,-0.734574,mutation_only
3,A2M,TCGA-06-5416-01,1,0,-1.254293,-1.089136,mutation_only
4,A2ML1,TCGA-06-5416-01,1,0,-1.399420,-3.763578,mutation_only
